# Proyecto Fase 2 — Riesgo de enfermedad coronaria en consulta cardiológica

**Ejemplo de referencia para BE3006 — Análisis de Datos Biomédicos.**

> Este notebook es un **dashboard ejecutivo**: solo orquesta llamadas a
> funciones del paquete `src/`. La lógica vive en los módulos. Si quieres
> entender una decisión, lee el módulo correspondiente.

| Item | Valor |
|---|---|
| Dataset | UCI Heart Disease — Cleveland (Detrano et al. 1989) |
| N inicial | 303 pacientes |
| Pregunta clínica | ¿Qué pacientes en consulta tienen riesgo elevado de enfermedad coronaria? |
| Decisión informada | Priorización para estudios diagnósticos invasivos (cateterismo) |
| Modelo | Regresión logística regularizada |

## 0. Decisión clínica a apoyar

**Contexto.** Un cardiólogo en consulta externa atiende a pacientes con dolor
torácico u otros síntomas. El cateterismo cardíaco es el estándar de oro para
diagnosticar enfermedad coronaria, pero es invasivo, costoso y conlleva riesgo.
La pregunta operativa es: **¿a qué pacientes priorizar para cateterismo
basándonos en la consulta inicial (historia, examen físico, ECG, prueba de
esfuerzo no invasiva)?**

**Decisión que el modelo informa**, no reemplaza:

- **No** decide solo a quién enviar. Es un score de apoyo.
- **Sí** ordena por probabilidad de enfermedad para que el especialista
  revise primero los casos de mayor riesgo.
- El umbral final lo decide el equipo clínico según costos relativos
  de falso negativo (paciente con enfermedad enviado a casa) vs. falso
  positivo (cateterismo innecesario). En este ejemplo usamos 0.5.

In [ ]:
import sys
from pathlib import Path

# Permitir importar src/ aunque el notebook se lance desde notebooks/
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import ingest, curate, eda, model, validate

## 1. Adquisición y Gobernanza

**Fuente.** UCI Machine Learning Repository, dataset *Heart Disease*,
procesado de la cohorte Cleveland Clinic Foundation. Los datos son de
1988, 303 pacientes con sospecha de enfermedad coronaria evaluados con
estudios no invasivos. Licencia: dominio público (CC0). Citación
obligatoria: Detrano et al., *Am J Cardiol* 1989; 64:304-310.

**Gobernanza para reproducibilidad.**

1. El CSV crudo vive en `data/raw/heart.csv` con su procedencia
   documentada en `data/README.md`.
2. La ingesta lo carga a SQLite (`db/heart.db`, tabla `raw_patients`)
   sin transformar — la curación es un paso separado y rastreable.
3. La base SQLite **no** se commitea: es regenerable. Esto fuerza a que
   cualquier resultado del análisis pueda reconstruirse desde el CSV.

In [ ]:
n = ingest.populate_db()
print(f"Cargadas {n} filas en raw_patients")
ingest.table_summary()

## 2. Preparación y Curación

**Decisiones explícitas tomadas en `src/curate.py`:**

1. **Faltantes.** El dataset codifica missing como `'?'`. Hay 6 pacientes con
   datos faltantes (4 en `ca`, 2 en `thal`). Decisión: **drop**, no
   imputación. Justificación: <2% de las filas, sin patrón sistémico
   observable, y la imputación en un dataset de 303 introduce ruido que no
   compensa la información ganada. N pasa de 303 → 297.

2. **Target binario.** La variable original `num` codifica severidad (0-4).
   Para la decisión clínica de priorización ("¿enviar a cateterismo?")
   un binario `target = 1 if num > 0 else 0` es suficiente. Documentamos
   que perdemos información de gradación.

3. **Tipos.** Numéricas a float; binarias y categóricas a int. Esto evita
   sorpresas en el preprocesamiento del pipeline de scikit-learn.

In [ ]:
summary = curate.curate()
print(
    f"Curación: {summary['raw_rows']} → {summary['curated_rows']} "
    f"(drop {summary['dropped_rows']} con NaN). "
    f"Tasa de positivos: {summary['positive_rate']:.1%}"
)
df = curate.read_curated()
df.head()

## 3. Análisis Exploratorio

Cada gráfico responde **una** pregunta. Si no se te ocurre la pregunta antes
de hacer el gráfico, no lo hagas.

### 3.1 ¿Está balanceado el outcome?

Importa porque define qué métricas reportar y si necesitamos compensar
el desbalance (class_weight, sampling).

In [ ]:
# Pregunta: ¿qué tan balanceado está el outcome?
fig = eda.plot_target_balance(df)
fig

### 3.2 Tabla descriptiva (Tabla 1)

Comparación de variables continuas entre grupos. Sin tests estadísticos en
este nivel — la validación cuantitativa va después.

In [ ]:
eda.descriptive_table(df)

### 3.3 Edad por estado de enfermedad

In [ ]:
# Pregunta: ¿la edad separa pacientes con vs. sin enfermedad coronaria?
eda.plot_age_by_target(df)

### 3.4 Tipo de dolor de pecho

In [ ]:
# Pregunta: ¿el tipo de dolor de pecho discrimina enfermedad?
# cp: 1=angina típica, 2=atípica, 3=no anginoso, 4=asintomático.
eda.plot_chest_pain_by_target(df)

### 3.5 Capacidad funcional vs. edad

In [ ]:
# Pregunta: ¿cómo se relaciona la frecuencia cardiaca máxima alcanzada
# en la prueba de esfuerzo (thalach) con la edad, separando por outcome?
eda.plot_thalach_vs_age(df)

### 3.6 Correlaciones entre numéricas

In [ ]:
# Pregunta: ¿qué features numéricas correlacionan con el outcome?
# Ojo: correlación lineal — no captura interacciones ni efectos no monótonos.
eda.plot_correlations(df)

## 4. Modelado

**Elección: regresión logística regularizada.**

- Coeficientes interpretables como log-odds → odds ratios para el equipo clínico.
- N=297 es chico. Modelos flexibles (Random Forest, gradient boosting)
  sobreajustan sin ganar AUC suficiente para justificar la opacidad.
- El curso valora **pipeline simple bien validado** sobre sofisticación
  injustificada. Si comparáramos con RF, lo haríamos honestamente vía CV
  contra esta baseline (no sustituirla).

**Pipeline de preprocesamiento dentro del modelo** (`src/model.py`):

1. `StandardScaler` para numéricas continuas.
2. Binarias pasan tal cual.
3. `OneHotEncoder(drop='first')` para categóricas (cp, restecg, slope, ca, thal).

**Anti-leakage.** La columna `num` (severidad 0-4) se excluye explícitamente.
`target` se derivó de `num`; usar `num` como feature filtraría la respuesta.

In [ ]:
trained = model.train(df, test_size=0.25, seed=42)
print(f"Train: {len(trained.X_train)} | Test: {len(trained.X_test)}")
print(f"Tasa positivos train: {trained.y_train.mean():.1%} | test: {trained.y_test.mean():.1%}")

**Coeficientes.** Para variables estandarizadas, `|coef|` permite comparar
fuerza relativa. `odds_ratio = exp(coef)`: cuántas veces se multiplican los
odds de enfermedad cuando la feature aumenta una unidad (o pasa de 0 a 1).

In [ ]:
model.coefficient_table(trained).head(10)

## 5. Validación

Reportamos **CV sobre el dataset completo** (estimación de generalización) y
**hold-out test** (medida del modelo final). No basta accuracy: en decisión
clínica con costos asimétricos importan sensibilidad, especificidad, VPP, VPN.

### 5.1 Cross-validation (5-fold estratificado, AUC)

In [ ]:
cv = validate.cross_validate_auc(df, n_splits=5, seed=42)
print(f"AUC CV: {cv['auc_mean']:.3f} ± {cv['auc_std']:.3f}")
print(f"Folds: {[round(x, 3) for x in cv['auc_folds']]}")

### 5.2 Hold-out test set

Métricas con umbral 0.5. Para una decisión real, el umbral debería elegirse
según los costos relativos de FP y FN (curva de Youden, cost-benefit, etc.).
Lo dejamos en 0.5 para mantener el ejemplo simple y trasladable; documentar
el umbral es la disciplina mínima.

In [ ]:
metrics, roc_fig = validate.evaluate(trained, threshold=0.5)
validate.format_metrics(metrics)

In [ ]:
roc_fig

In [ ]:
validate.plot_confusion_matrix(metrics)

## 6. Limitaciones y decisión informada

### Lo que este modelo NO es

- **No es un diagnosticador.** Es un score de priorización. La decisión
  clínica final descansa en el cardiólogo con la historia completa.
- **No generaliza al 2026 sin validación.** Datos de Cleveland 1988. La
  prevalencia, el perfil demográfico y los criterios de derivación de hoy
  difieren. Validar prospectivamente antes de cualquier uso clínico real.
- **N pequeño.** 297 pacientes después de curación. Los intervalos de
  confianza alrededor de AUC, sensibilidad y especificidad son anchos
  (la desviación de CV es ~0.05). En un dataset de 30 000 pacientes la
  conclusión podría ser distinta.

### Sesgos conocidos

- **Centro único.** Todos los pacientes de Cleveland Clinic. Sesgo de
  espectro: sobre-representación de pacientes referidos (más enfermos
  que la población general en consulta primaria).
- **Demográfico.** Cohorte estadounidense de 1988, mayoritariamente
  masculina (~68%). Extender a poblaciones latinoamericanas requiere
  validación local.
- **Outcome.** El target binario colapsa severidad. Un paciente con
  obstrucción de 1 vaso y otro con 4 vasos cuentan igual aquí.

### Recomendación operativa (basada en este modelo)

- AUC test ≈ 0.91, sensibilidad 80%, especificidad 87.5% con umbral 0.5.
- **Si se priorizara cateterismo por orden de score, el modelo identificaría
  4 de cada 5 pacientes con enfermedad antes que los demás.** Eso es
  utilidad clínica concreta.
- Las features con mayor peso (`ca`, `thal`, `cp`) son consistentes con
  conocimiento cardiológico (vasos visibles, defecto talámico, tipo de
  dolor) — buena señal de cara a aceptación clínica.

### Próximos pasos honestos

1. Validar en una cohorte contemporánea local (UVG / hospitales aliados).
2. Calibrar el umbral con costos clínicos reales (no usar 0.5 por defecto).
3. Comparar contra el score clínico estándar (Diamond-Forrester, p.ej.).
4. Reentrenar con datos longitudinales si están disponibles.